# Segundo Project Brief: Modelo Predictivo de Detección de Fraude Transaccional

## 0. Executive Summary
* **La pregunta:** ¿Es posible predecir qué transacciones son fraudulentas en tiempo real basándose en el importe, la hora y los componentes de comportamiento para priorizar las revisiones del equipo de operaciones?
* **El modelo:** Un pipeline de Machine Learning utilizando un clasificador optimizado (`Random Forest` / `XGBoost`) con preprocesamiento integrado.
* **Mejora frente a la regla de hoy:** Supera el rendimiento de la regla estática actual reduciendo drásticamente los falsos positivos y maximizando la detección de fraudes reales.
* **Acción para el stakeholder:** El supervisor de fraude utilizará la lista priorizada de alertas para bloquear o auditar únicamente el top de riesgo operativo cada jornada.
* **Límite principal:** El desbalanceo extremo de clases requiere un ajuste fino del umbral de decisión para evitar fricción innecesaria a clientes legítimos.

## 1. From the Brief to an ML Task

| Pregunta clave | Definición en el proyecto |
|---|---|
| ¿Qué decisión alimenta? | Qué transacciones congelar o auditar prioritariamente al inicio del turno operativo. |
| Tipo de tarea | Clasificación binaria (Supervisado). |
| Target exacto | Columna `Class` (0 = Legítima, 1 = Fraude). |
| Grano de la predicción | Una transacción bancaria individual. |
| Momento de predicción | En el momento exacto en que se ejecuta la transacción en la pasarela de pago. |
| Características disponibles | `Time`, `Amount` y los componentes principales `V1` a `V28` disponibles en el *stream*. |
| Regla de hoy (Baseline) | Reglas estáticas basadas en importes fijos y alertas manuales aleatorias. |
| Métrica principal | Recall (Sensibilidad) y Precision-Recall AUC (PR-AUC) debido al desbalanceo de clases. |
| Traducción de negocio | Minimizar pérdidas monetarias por fraude sin superar el límite de falsos positivos en clientes. |

In [2]:
# Importación de librerías esenciales
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, auc

# Fijar semilla para garantizar reproducibilidad
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Librerías importadas correctamente y semilla fijada.")

Librerías importadas correctamente y semilla fijada.


In [5]:
# Carga del dataset utilizando ruta relativa
import os

data_path = "creditcard.csv" if os.path.exists("creditcard.csv") else "project/creditcard.csv"
df = pd.read_csv(data_path)

print(f"Dataset cargado con éxito. Dimensiones: {df.shape[0]} filas y {df.shape[1]} columnas.")

# Resumen rápido de calidad
print(f"Valores nulos en el dataset: {df.isnull().sum().sum()}")
print(f"Duplicados eliminados: {df.duplicated().sum()}")
df = df.drop_duplicates()

Dataset cargado con éxito. Dimensiones: 23858 filas y 31 columnas.
Valores nulos en el dataset: 9
Duplicados eliminados: 88


## 3. What the Exploration Showed
* **Distribución del Target:** Se confirma un desbalanceo extremo (aprox. 0.17% de fraudes frente a 99.83% de transacciones normales), lo que valida el uso de métricas orientadas a clases minoritarias.
* **Hipótesis H1:** *Refutada parcialmente en formato estático, pero muestra correlación horaria en importes altos.*
* **Hipótesis H2:** *Soportada.* Los fraudes se concentran significativamente en los extremos de la variable `Amount`.

In [8]:
# Eliminar filas donde 'Class'  y/o cualquier caracteristica tenga valores nulos
df_clean = df.dropna(subset=['Class']).copy()

# Rellenar o eliminar nulos en X si los hubiera
X = df_clean.drop(columns=['Class']).fillna(0)
y = df_clean['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Train set: {X_train.shape[0]} muestras")
print(f"Test set: {X_test.shape[0]} muestras")

Train set: 116876 muestras
Test set: 29219 muestras


In [9]:
from sklearn.ensemble import RandomForestClassifier

# Creación del Pipeline que integra el escalador y el modelo
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1))
])

# Entrenamiento con sus datos
print("Entrenando el modelo...")
pipeline.fit(X_train, y_train)

# Evaluacion con test :D
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

# Guardamos el modelo final
os.makedirs("project", exist_ok=True)
model_path = "project/model.joblib"
joblib.dump(pipeline, model_path)
print(f"Modelo guardado exitosamente en: {model_path}")

Entrenando el modelo...
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00     29164
         1.0       0.98      0.73      0.83        55

    accuracy                           1.00     29219
   macro avg       0.99      0.86      0.92     29219
weighted avg       1.00      1.00      1.00     29219

Modelo guardado exitosamente en: project/model.joblib
